## Frontend

In [ ]:
import streamlit as st
from LLM import get_entities, generate_response
from Neo4j import Neo4jHandler

In [ ]:
# Streamlit app title
st.title("Knowledge Graph-Powered Chat: Query Your Dataset")

# Initialize session state to store chat history and user input
if "messages" not in st.session_state:
    st.session_state["messages"] = []

if "user_input" not in st.session_state:
    st.session_state["user_input"] = ""

- We initialize the Neo4j handler by providing the required credentials to connect to the Neo4j database.

In [ ]:
neo4j_handler = Neo4jHandler(connection_uri, username, password)

In [ ]:
user_input = st.text_input("Ask a question related to your dataset:", value=st.session_state["user_input"])

In [ ]:
if st.button("Send") and user_input:
  # Step 1: Extract entities from the user query
  extracted_entities = get_entities(user_input)
  print("Entities in the query: ", extracted_entities)
  
  # Step 2: Retrieve associated relationships from the Neo4j knowledge graph
  related_relationships_tuple_list = neo4j_handler.get_entities_and_relationships(extracted_entities)
  print("Extracted context from the Knowledge Graph: ", related_relationships_tuple_list)
  
  # Step 3: Augment the query with knowledge graph context
  messages = [
    {"role": "system", "content": f"Use the following knowledge graph context (provided as relationship tuple list: (Entity 1, Relation, Entity 2)) to answer the user queries.\n{related_relationships_tuple_list}"},
    {"role": "user", "content": user_input},
  ]
  
  # Step 4: Get GPT-4 response using the augmented query
  bot_response = generate_response(messages)
  
  # Step 5: Append the user message and bot response to the session state chat history
  st.session_state["messages"].append({"role": "user", "content": user_input})
  st.session_state["messages"].append({"role": "assistant", "content": bot_response})

  # Clear the input field in session state
  st.session_state["user_input"] = ""

### UI

In [ ]:
def display_message(message, is_user):
  if is_user:
    st.markdown(f"""
    <div style="background-color:#DCF8C6;padding:10px;border-radius:10px;margin-bottom:10px;">
    {message}
    </div>
    """, unsafe_allow_html=True)
  else:
    st.markdown(f"""
    <div style="background-color:#E6E6FA;padding:10px;border-radius:10px;margin-bottom:10px;">
    {message}
    </div>
    """, unsafe_allow_html=True)

- We iterate through the saved chat history (stored in the session state) and display the messages in the respective colored boxes based on whether they are from the user or the assistant.

In [ ]:
if st.session_state["messages"]:
  for msg in st.session_state["messages"]:
    if msg["role"] == "user":
      display_message(msg["content"], is_user=True)
    elif msg["role"] == "assistant":
      display_message(msg["content"], is_user=False)

- Close the Neo4j driver connection to maintain the application’s efficiency by releasing unused connections and avoiding memory leaks and connection limits

In [ ]:
neo4j_handler.close()

## LLM.py

-  extract entities from the query

In [ ]:
def get_entities(query):
  content = extract_entities(query)
  entities = parse_llm_response_content(content)
  return entities

- use the LLM to generate a response to the query.

In [ ]:
def generate_response(messages):
  try:
    response = client.chat.completions.create(
    model="gpt-4",
    messages=messages,
    )
    response_content = response.choices[0].message.content
    return response_content
  except Exception as e:
    return f"Error: {str(e)}"

## Neo4j.py

- __init__() method: This constructor initializes a connection to the Neo4j database using the provided URI, username, and password. It creates a driver instance that will be used to interact with the database.

- close() method: This method safely closes the connection to the database. It’s important to call this when we are done using the Neo4jHandler instance to free up resources.

In [ ]:
from neo4j import GraphDatabase

In [ ]:
class Neo4jHandler:
  def __init__(self, connection_uri, username, password):
    self.driver = GraphDatabase.driver(connection_uri, auth=(username, password))
  
  def close(self):
    self.driver.close()
  
  def get_entities_and_relationships(self, entities):
    with self.driver.session() as session:
      results = []
      for entity in entities:
        result = session.read_transaction(self._retrieve_relationships, entity)
        results.extend(result)
      return results

In [ ]:
 @staticmethod
  def _retrieve_relationships(tx, entity_name):
    query = """
    MATCH (e1:Entity {name: $entity_name})-[r:RELATION]->(e2:Entity)
    RETURN e1.name AS entity1, r.type AS relationship, e2.name AS entity2
    """
    result = tx.run(query, entity_name=entity_name)
    return [(record["entity1"], record["relationship"], record["entity2"]) for record in result]    

- get_entities_and_relationships() method: This method takes a list of entity names as input. For each entity, it executes a read transaction to retrieve relationships related to that entity. The results for all entities are combined into a single list and returned.

- _retrieve_relationships() static method: This method constructs and executes a Cypher query to find relationships for a given entity. The query matches an entity node (e1) and retrieves all outgoing relationships (r) leading to other entity nodes (e2). It returns a list of tuples, each containing the names of the entities and the type of relationship between them.